In [11]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor
import numpy as np
from joblib import dump
import sys
import os

In [12]:
sys.path.append(os.path.abspath(".."))

from Data_Preparation.soil_temp_data_preparation import (
    create_soil_temp_pipeline,
    prepare_soil_temp_data
)

In [13]:
def evaluate_xgb(X_train, y_train, X_dev, y_dev):
    print("Evaluating XGBoost Regressor...")

    param_grid = {
        'algo__n_estimators': [100, 300, 500, 1000],
        'algo__max_depth': [3, 5],
        'algo__learning_rate': [0.05, 0.1],
        'algo__subsample': [0.8, 1.0],
    }

    pipeline = create_soil_temp_pipeline()

    pipeline_with_algo = Pipeline(steps=[
        ('preprocessor', pipeline),
        ('algo', XGBRegressor(objective='reg:squarederror', random_state=42))
    ])

    grid_search = GridSearchCV(
        pipeline_with_algo,
        param_grid,
        cv=3,
        scoring='neg_mean_squared_error',
        verbose=1,
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)
    best_estimator = grid_search.best_estimator_

    try:
        model = best_estimator.named_steps["algo"]
        feature_names = X_train.columns
        importances = model.feature_importances_

        feature_df = pd.DataFrame({
            "Feature": feature_names,
            "Importance": importances
        }).sort_values(by="Importance", ascending=False)

        print("\nTop 10 Most Important Features:")
        print(feature_df.head(10))

    except Exception as e:
        print("Could not extract feature importances:", e)

    y_pred = best_estimator.predict(X_dev)
    rmse = np.sqrt(mean_squared_error(y_dev, y_pred))
    mae = mean_absolute_error(y_dev, y_pred)
    r2 = r2_score(y_dev, y_pred)

    print("Grid searching is done!")
    print("Best score (neg MSE):", grid_search.best_score_)
    print("Best hyperparameters:")
    print(grid_search.best_params_)

    return best_estimator, rmse, mae, r2

In [14]:
def evaluate_metrics(y_true, y_pred, label):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mean_target = np.mean(y_true)

    print(f"\n📊 {label} Set Performance:")
    print(f"Mean of y_{label.lower()}: {mean_target:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"R²: {r2:.4f}")

    return rmse, mae, r2

In [15]:
print("\n🚀 Evaluating model for: Soil Temperature")

X_train, X_dev, X_test, y_train, y_dev, y_test = prepare_soil_temp_data()


🚀 Evaluating model for: Soil Temperature


In [16]:
best_model, _, _, _ = evaluate_xgb(X_train, y_train, X_dev, y_dev)

Evaluating XGBoost Regressor...
Fitting 3 folds for each of 32 candidates, totalling 96 fits

Top 10 Most Important Features:
                Feature  Importance
2       air_temperature    0.952708
5                 month    0.027239
3  dewpoint_temperature    0.005528
6                   day    0.004482
1             longitude    0.003456
4   total_precipitation    0.003348
0              latitude    0.003239
Grid searching is done!
Best score (neg MSE): -0.023171136421523172
Best hyperparameters:
{'algo__learning_rate': 0.1, 'algo__max_depth': 5, 'algo__n_estimators': 1000, 'algo__subsample': 0.8}


In [17]:
print("✅ Data Split Shapes:")
print("  X_train:", X_train.shape)
print("  X_dev:", X_dev.shape)
print("  X_test:", X_test.shape)
print("  y_train:", y_train.shape)
print("  y_dev:", y_dev.shape)
print("  y_test:", y_test.shape)

✅ Data Split Shapes:
  X_train: (1679864, 7)
  X_dev: (419967, 7)
  X_test: (10000, 7)
  y_train: (1679864,)
  y_dev: (419967,)
  y_test: (10000,)


In [18]:
y_train_pred = best_model.predict(X_train)
y_dev_pred = best_model.predict(X_dev)
y_test_pred = best_model.predict(X_test)

In [19]:
evaluate_metrics(y_train, y_train_pred, "Train")
evaluate_metrics(y_dev, y_dev_pred, "Dev")
evaluate_metrics(y_test, y_test_pred, "Test")


📊 Train Set Performance:
Mean of y_train: -0.0002
RMSE: 0.1516
MAE: 0.1123
R²: 0.9770

📊 Dev Set Performance:
Mean of y_dev: 0.0007
RMSE: 0.1521
MAE: 0.1127
R²: 0.9768

📊 Test Set Performance:
Mean of y_test: 0.0005
RMSE: 0.1525
MAE: 0.1132
R²: 0.9774


(0.15251634583302845, 0.11316085827472923, 0.9773927333545794)

In [20]:
dump(best_model, "../models/soil_temperature_model.joblib")

print("✅ Model saved as: ../models/soil_temperature_model.joblib")

✅ Model saved as: ../models/soil_temperature_model.joblib
